In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
from google.colab import drive
from tensorflow.keras.models import load_model


model_path = '/content/drive/MyDrive/CAS Advanced Machine Learning/Luftbild_Colorization/Retrained_HyperUNet_RMSE.h5'
model = load_model(model_path)

weights = model.get_weights()

# Confirm the model is loaded
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, None, None, 1)]      0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, None, None, 64)       640       ['input_1[0][0]']             
                                                                                                  
 conv2d_1 (Conv2D)           (None, None, None, 64)       36928     ['conv2d[0][0]']              
                                                                                                  
 max_pooling2d (MaxPooling2  (None, None, None, 64)       0         ['conv2d_1[0][0]']            
 D)                                                                                           

In [24]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate
from tensorflow.keras.optimizers import Adam

In [23]:
def unet1(input_size):
    inputs = Input(input_size)  # 0
    # layers 1-5
    conv1 = Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(inputs) # 1
    conv1 = Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv1)  # 2
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1) # 3
    # layers 4-6
    conv2 = Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool1) # 4
    conv2 = Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv2) # 5
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2) # 6
    # layers 7-10
    conv3 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool2) # 7
    conv3 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv3) # 8
    conv3 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv3) # 9
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3) # 10
    # layers 11-14
    conv4 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool3) # 11
    conv4 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv4) # 12
    conv4 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv4) # 13
    pool4 = MaxPooling2D(pool_size=(2, 2))(conv4) # 14
    # layers 15-18
    conv5 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool4) # 15
    conv5 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv5) # 16
    conv5 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv5) # 17
    pool5 = MaxPooling2D(pool_size=(2, 2))(conv5) # 18

    # layers 19-21
    conv55 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool5) # 19
    conv55 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv55) # 20
    conv55 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv55) # 21

    # layers 22-26
    up66 = Conv2D(512, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(conv55)) # 22+23
    merge66 = concatenate([conv5,up66], axis = 3) # 24
    conv66 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge66) # 25
    conv66 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv66) # 26

    # layers 27-31
    up6 = Conv2D(512, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(conv66))# 27+28
    merge6 = concatenate([conv4,up6], axis = 3) # 29
    conv6 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge6) # 30
    conv6 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv6) # 31

    # layers 32-36
    up7 = Conv2D(256, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(conv6)) # 32+33
    merge7 = concatenate([conv3,up7], axis = 3) # 34
    conv7 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge7) # 35
    conv7 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv7) # 36

    # layers 37-41
    up8 = Conv2D(128, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(conv7)) # 37+38
    merge8 = concatenate([conv2,up8], axis = 3) # 39
    conv8 = Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge8) # 40
    conv8 = Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv8) # 41

    # layers 42-46
    up9 = Conv2D(64, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(conv8)) # 42+43
    merge9 = concatenate([conv1,up9], axis = 3) # 44
    conv9 = Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge9) # 45
    conv9 = Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv9) # 46

    # Up_f01 = conv1
    # Up_f02 = UpSampling2D(size = (2,2))(conv2)
    # Up_f03 = UpSampling2D(size = (4,4))(conv3)
    # Up_f04 = UpSampling2D(size = (8,8))(conv4)
    # Up_f05 = UpSampling2D(size = (16,16))(conv5)
    # Up_f06 = UpSampling2D(size = (32,32))(conv55)
    # Up_f16 = UpSampling2D(size = (32,32))(conv66)
    # Up_f15 = UpSampling2D(size = (16,16))(conv6)
    # Up_f14 = UpSampling2D(size = (8,8))(conv7)
    # Up_f13 = UpSampling2D(size = (4,4))(conv8)
    # Up_f12 = UpSampling2D(size = (2,2))(conv9)
    # Up_f11 = conv10

    Up_f01 = conv1     #
    Up_f02 = UpSampling2D(size = (2,2))(conv2) # 47
    # Up_f03 = UpSampling2D(size = (4,4))(conv3) # 49
    # Up_f04 = UpSampling2D(size = (8,8))(conv4) # 40
    # Up_f05 = UpSampling2D(size = (16,16))(conv5) # 50
    # Up_f06 = UpSampling2D(size = (32,32))(conv55) # 51
    # Up_f15 = UpSampling2D(size = (16,16))(conv66) # 52
    # Up_f14 = UpSampling2D(size = (8,8))(conv6) # 53
    # Up_f13 = UpSampling2D(size = (4,4))(conv7) # 54
    Up_f12 = UpSampling2D(size = (2,2))(conv8) # 48
    # Up_f12 = UpSampling2D(size = (2,2))(conv9)
    Up_f11 = conv9  #  conv10     # 56

    # merge11 = concatenate([inputs,Up_f01,Up_f11,Up_f02,Up_f12,Up_f03,Up_f13,Up_f04,Up_f14,Up_f05,Up_f15,Up_f06,Up_f16], axis = 3)
    # merge11 = concatenate([Up_f01,Up_f11,Up_f02,Up_f12,Up_f03,Up_f13], axis = 3)
    merge11 = concatenate([Up_f01,Up_f11,Up_f02,Up_f12], axis = 3) # 49

    conv11 = Conv2D(128, 3, activation = 'relu', padding = 'same',)(merge11) # 50

    conv12 = Conv2D(64, 3, activation = 'relu', padding = 'same',)(conv11) # 51

    conv13 = Conv2D(64, 3, activation = 'relu', padding = 'same',)(conv12) # 52

    conv14 = Conv2D(2, 3, activation = 'tanh', padding = 'same',)(conv13) # 53
    model = Model(inputs, conv14)

    # model.compile(optimizer = Adam(lr = 1e-4), loss = 'mean_squared_error', metrics = ['accuracy'])
    model.compile(optimizer = Adam(lr = 1e-4), loss = 'mean_absolute_error', metrics = ['RootMeanSquaredError'])
    return model

model = Model(inputs=input_layer, outputs=output_layer)

model.summary()

Model: "model_5"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, None, None, 1)]      0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, None, None, 64)       640       ['input_1[0][0]']             
                                                                                                  
 conv2d_1 (Conv2D)           (None, None, None, 64)       36928     ['conv2d[0][0]']              
                                                                                                  
 max_pooling2d (MaxPooling2  (None, None, None, 64)       0         ['conv2d_1[0][0]']            
 D)                                                                                         

In [28]:
# Define the grayscale input size
input_size = (256, 256, 1)  # 1 channel for grayscale
new_model = unet1(input_size)

In [30]:
# Load the pre-trained model
model_path = '/content/drive/MyDrive/CAS Advanced Machine Learning/Luftbild_Colorization/Retrained_HyperUNet_RMSE.h5'
model_A = load_model(model_path)

# Ensure model_A and new_model architectures are identical before setting weights
new_model.set_weights(model_A.get_weights())

In [31]:
# Print weight shapes for model_A
print("Model_A weights shapes:")
for layer in model_A.layers:
    weights = layer.get_weights()
    print(f"{layer.name}: {[w.shape for w in weights]}")

# Print weight shapes for new_model
print("New_Model weights shapes:")
for layer in new_model.layers:
    weights = layer.get_weights()
    print(f"{layer.name}: {[w.shape for w in weights]}")


Model_A weights shapes:
input_1: []
conv2d: [(3, 3, 1, 64), (64,)]
conv2d_1: [(3, 3, 64, 64), (64,)]
max_pooling2d: []
conv2d_2: [(3, 3, 64, 128), (128,)]
conv2d_3: [(3, 3, 128, 128), (128,)]
max_pooling2d_1: []
conv2d_4: [(3, 3, 128, 256), (256,)]
conv2d_5: [(3, 3, 256, 256), (256,)]
conv2d_6: [(3, 3, 256, 256), (256,)]
max_pooling2d_2: []
conv2d_7: [(3, 3, 256, 512), (512,)]
conv2d_8: [(3, 3, 512, 512), (512,)]
conv2d_9: [(3, 3, 512, 512), (512,)]
max_pooling2d_3: []
conv2d_10: [(3, 3, 512, 512), (512,)]
conv2d_11: [(3, 3, 512, 512), (512,)]
conv2d_12: [(3, 3, 512, 512), (512,)]
max_pooling2d_4: []
conv2d_13: [(3, 3, 512, 512), (512,)]
conv2d_14: [(3, 3, 512, 512), (512,)]
conv2d_15: [(3, 3, 512, 512), (512,)]
up_sampling2d: []
conv2d_16: [(2, 2, 512, 512), (512,)]
concatenate: []
conv2d_17: [(3, 3, 1024, 512), (512,)]
conv2d_18: [(3, 3, 512, 512), (512,)]
up_sampling2d_1: []
conv2d_19: [(2, 2, 512, 512), (512,)]
concatenate_1: []
conv2d_20: [(3, 3, 1024, 512), (512,)]
conv2d_21: [(3

In [32]:
for layer in new_model.layers:
    print(f"{layer.name}: {layer.get_weights()}")

for layer in model_A.layers:
    print(f"{layer.name}: {layer.get_weights()}")


Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
       -1.20346877e-03,  8.74476284e-02, -2.85982993e-02, -8.95059556e-02,
        4.89687175e-02,  8.11026841e-02,  4.50396501e-02, -3.23279854e-03,
        9.67835262e-02,  7.46315718e-02, -1.71464365e-02, -4.31927480e-02,
        4.93165925e-02,  5.40956594e-02,  8.98150504e-02,  5.83661161e-02,
       -3.21832187e-02,  8.62189531e-02,  5.20257317e-02,  4.60258722e-02,
       -3.06159202e-02,  4.48649898e-02, -2.68396977e-02,  5.16757704e-02,
       -3.18890586e-02, -1.74862356e-03, -5.46206301e-03,  6.06379993e-02,
        6.32781014e-02,  8.69966205e-03, -7.64595792e-02, -1.25408480e-02,
        7.70841911e-03, -3.21729369e-02, -8.26127175e-03,  6.07952438e-02,
        1.00223586e-01, -1.64940134e-02,  9.74879861e-02, -2.38527805e-02,
        7.40591111e-03,  1.27082272e-02,  4.84514935e-03,  3.75864953e-02,
        8.95686448e-02,  6.42127823e-03,  6.77845776e-02,  8.57410654e-02,
       -3.71361431e-03, -2.694814

In [33]:
# Save the new model to an HDF5 file
new_model_path = '/content/drive/MyDrive/CAS Advanced Machine Learning/Luftbild_Colorization/New_Unet_Model.h5'
new_model.save(new_model_path)

print(f"Model saved to {new_model_path}")


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Model saved to /content/drive/MyDrive/CAS Advanced Machine Learning/Luftbild_Colorization/New_Unet_Model.h5
